# 🔁 Pseudo-C2 pose-flip check (cryoSPARC J486 vs J487)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab_pseudo_c2_check.ipynb)

About 1 in 7 particles have J486 and J487 poses that differ by ~180°. J486 (Reference-Based Motion
Correction) passes its alignments through unchanged from J485, so this compares the **J485** and **J487**
refinements. cryoDRGN's `pose.pkl` was parsed from **J487**, so every run (00, 01, 03), back-projection
and landscape used J487 poses.

This one cell asks three questions:

1. **Is it really a pseudo-C2 protomer swap?** It matches particles by `uid`, not by row: the files have
   419,954 vs 418,418 rows, and a row-by-row comparison drifts after the first missing particle, giving
   spurious large angles. A genuine swap gives the *same* 180° operator about *one* axis in the map frame
   for every flipped particle. Random misalignment gives scattered axes.
2. **Which particles?** It writes flipped / unflipped index files in cryoDRGN's global numbering, ready for
   `--ind` in the back-projection notebook. It also writes a J486 pose file in the same order, so the flipped
   subset can be reconstructed once with each job's poses.
3. **Did it leak into the cryoDRGN results?** It tests whether flipped particles are concentrated in any
   UMAP region, latent direction, volume PC, landscape state, micrograph or viewing direction.

**Runtime:** CPU is enough, and no cryoDRGN install is needed. It reads two ~230 MB `.cs` files from Drive
and takes about 1–2 minutes plus Drive read time. It needs roughly 1–2 GB of RAM. Results go to `OUT_DIR`:
`pc2_report.txt` plus PNGs, CSVs and the index `.pkl` files.

**Reading the result:** the verdict block at the end says whether the flips share one map-frame axis
(pseudo-C2) and whether they correlate with anything in cryoDRGN. A *null* correlation does not mean the
flips are harmless. Flipped particles see the same encoder input as correctly posed ones, so they mostly
put ~14% "ghost" density of any asymmetric feature on the partner protomer instead of forming their own
cluster.


In [ ]:
#@title Pseudo-C2 flip check: J486 vs J487 cryoSPARC poses { display-mode: "form" }
# -----------------------------------------------------------------------------------------
# Question: about 1 in 7 particles have J486 and J487 poses that differ by ~180 deg. Is that a
# pseudo-C2 "protomer swap" of the MINA53 dimer (the same particle aligned with its two
# protomers exchanged), or something else?
#
# What this cell does (numpy + matplotlib only; scipy not needed):
#   1. loads both .cs files and matches rows by 'uid' (falls back to blob path+idx),
#   2. computes each particle's relative rotation between the two jobs, in the MAP (body)
#      frame and in the IMAGE (lab) frame, plus the shift change (J486's shifts are J485 passthrough values;
#      J486 recentred the particles, so the shift reference is detected first, see J486_SHIFT_MODE),
#   3. classifies "flipped" = relative angle > THRESH_DEG and reports fractions,
#   4. tests whether the flips share ONE rotation axis (a real pseudo-2-fold gives one tight
#      axis in the map frame) and prints that axis,
#   5. writes cryoDRGN-numbered index files (global index = J487 row = cryoDRGN stack index),
#   6. checks whether flips line up with the cryoDRGN UMAP, latent z, volume PCs, landscape states,
#      micrographs, viewing direction and per-particle alignment scores,
#   7. prints a verdict.
# Every output goes to OUT_DIR. Inputs are only read. Leave an optional path empty to skip that part.
# -----------------------------------------------------------------------------------------

# ---- Required inputs ----
J486_CS = "/content/drive/MyDrive/mina53_mpp6/J486_particles_0/J486_particles_0_exported.cs"  #@param {type:"string"}
J487_CS = "/content/drive/MyDrive/mina53_mpp6/J487_particles/J487_particles_exported.cs"  #@param {type:"string"}
IND_PKL = "/content/drive/MyDrive/mina53_mpp6/cleaned_particle_sets/cleaned_full_set_405158.pkl"  #@param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/mina53_mpp6/pseudo_c2_check"  #@param {type:"string"}
# ---- Optional inputs (filtered order = rows of the --ind run; leave empty to skip) ----
UMAP_PKL = "/content/drive/MyDrive/mina53_mpp6/01_cryodrgn128_filtered/analyze.50/umap.pkl"  #@param {type:"string"}
Z_PKL = "/content/drive/MyDrive/mina53_mpp6/01_cryodrgn128_filtered/z.50.pkl"  #@param {type:"string"}
VOLPCA_ALL = "/content/drive/MyDrive/mina53_mpp6/landscape/01_cryodrgn128_filtered.50/landscape_full/vol_pca_all.pkl"  #@param {type:"string"}
LANDSCAPE_STATES_DIR = "/content/drive/MyDrive/mina53_mpp6/landscape_states/01_cryodrgn128_filtered.50_ward10"  #@param {type:"string"}
POSE_PKL = "/content/drive/MyDrive/mina53_mpp6/pose.pkl"  #@param {type:"string"}
# ---- Options ----
THRESH_DEG = 150.0  #@param {type:"number"}
REMOVE_GLOBAL_OFFSET = True  #@param {type:"boolean"}
WRITE_J486_POSE_PKL = True  #@param {type:"boolean"}
UMAP_KMEANS_K = 6  #@param {type:"integer"}
# J486 is Reference-Based Motion Correction with "recenter particles" on: its alignments3D (incl. shifts) are
# passed through from J485 and refer to the OLD particle centring, while its blobs (= the cryoDRGN stack) were
# re-extracted recentred. "auto" picks the J486 shift reference that agrees with J487 on the non-flipped
# particles: as_is (shifts valid for the stack), zero (exact recentring) or frac (integer-pixel recentring).
J486_SHIFT_MODE = "auto"  #@param ["auto", "as_is", "zero", "frac"]

import os, sys, glob, json, pickle, time
import numpy as np

# Testing hook: environment variables named PC2_<PARAM> override the form values above.
# Colab never sets them, so you can ignore this.
for _name in ("J486_CS", "J487_CS", "IND_PKL", "OUT_DIR", "UMAP_PKL", "Z_PKL", "VOLPCA_ALL",
              "LANDSCAPE_STATES_DIR", "POSE_PKL", "THRESH_DEG", "REMOVE_GLOBAL_OFFSET",
              "WRITE_J486_POSE_PKL", "UMAP_KMEANS_K", "J486_SHIFT_MODE"):
    _v = os.environ.get("PC2_" + _name)
    if _v is not None:
        _old = globals()[_name]
        globals()[_name] = (_v.strip().lower() in ("1", "true", "yes")) if isinstance(_old, bool) else type(_old)(_v)

_IN_COLAB = "google.colab" in sys.modules
import matplotlib
if not _IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

if _IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Colours (reference categorical slots 1-2 and neutral ink)
C_UNF, C_FLIP, C_INK, C_MUTED, C_GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#d9d8d4"
_REPORT = []


def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    _REPORT.append(s)


def section(title):
    log("")
    log("=" * 88)
    log(title)
    log("=" * 88)


def load_pkl(path):
    try:
        from cryodrgn import utils as _cu  # use cryoDRGN's loader when available
        return _cu.load_pkl(path)
    except Exception:
        with open(path, "rb") as f:
            return pickle.load(f)


def save_pkl(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)


def load_cs(path):
    try:
        return np.load(path)
    except ValueError:  # object fields in some .cs exports
        return np.load(path, allow_pickle=True)


def field(cs, name):
    return np.asarray(cs[name]) if name in (cs.dtype.names or ()) else None


def opt_path(p):
    return bool(p) and str(p).strip() != "" and os.path.exists(p)


def savefig(fig, name):
    p = os.path.join(OUT_DIR, name)
    fig.savefig(p, dpi=130, bbox_inches="tight")
    if _IN_COLAB:
        plt.show()
    plt.close(fig)
    log(f"  [png] {p}")


def style_ax(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(C_MUTED)
    ax.tick_params(colors=C_MUTED, labelsize=8)
    ax.grid(True, color=C_GRID, lw=0.6, alpha=0.7)
    ax.set_axisbelow(True)


# ------------------------------- rotation helpers -------------------------------
def skew(v):
    K = np.zeros(v.shape[:-1] + (3, 3))
    K[..., 0, 1], K[..., 0, 2] = -v[..., 2], v[..., 1]
    K[..., 1, 0], K[..., 1, 2] = v[..., 2], -v[..., 0]
    K[..., 2, 0], K[..., 2, 1] = -v[..., 1], v[..., 0]
    return K


def expmap(v):
    """Rodrigues: axis-angle vector (N,3) [rad] -> rotation matrix (N,3,3); same as cryodrgn lie_tools.expmap."""
    v = np.asarray(v, np.float64).reshape(-1, 3)
    th = np.linalg.norm(v, axis=1)
    R = np.empty((len(v), 3, 3))
    nz = th > 1e-10
    I = np.eye(3)
    if nz.any():
        K = skew(v[nz] / th[nz, None])
        R[nz] = I + np.sin(th[nz])[:, None, None] * K + (1.0 - np.cos(th[nz]))[:, None, None] * (K @ K)
    if (~nz).any():  # zero angle: first-order expansion (exactly I at 0)
        R[~nz] = I + skew(v[~nz])
    return R


def T(R):
    return np.swapaxes(R, -1, -2)


def angle_axis(R):
    """Rotation angle [deg] and unit axis for (N,3,3) matrices, numerically robust from 0 to 180 deg."""
    v = 0.5 * np.stack([R[:, 2, 1] - R[:, 1, 2], R[:, 0, 2] - R[:, 2, 0], R[:, 1, 0] - R[:, 0, 1]], -1)  # sin(th) * u
    s = np.linalg.norm(v, axis=1)
    c = np.clip(0.5 * (np.trace(R, axis1=1, axis2=2) - 1.0), -1.0, 1.0)
    th = np.arctan2(s, c)
    ax = np.zeros_like(v)
    small = (c >= 0) & (s > 1e-12)  # th < 90 deg: antisymmetric part is well conditioned
    ax[small] = v[small] / s[small, None]
    big = c < 0  # th > 90 deg: use symmetric part, (S - cos I)/(1 - cos) = u u^T (the "(R+I)/2" columns at th = pi)
    if big.any():
        S = 0.5 * (R[big] + T(R[big]))
        M = (S - c[big, None, None] * np.eye(3)) / (1.0 - c[big])[:, None, None]
        d = np.einsum("nii->ni", M)
        k = np.argmax(d, axis=1)
        idx = np.arange(len(k))
        a = M[idx, :, k] / np.sqrt(np.clip(d[idx, k], 1e-15, None))[:, None]
        a /= np.linalg.norm(a, axis=1, keepdims=True)
        sg = np.sign(np.einsum("ni,ni->n", a, v[big]))
        sg[sg == 0] = 1.0
        ax[big] = a * sg[:, None]
    return np.degrees(th), ax


def proj_so3(M):
    U, _, Vt = np.linalg.svd(M)
    D = np.diag([1.0, 1.0, np.sign(np.linalg.det(U @ Vt))])
    return U @ D @ Vt


def axial_dev(A, e):
    return np.degrees(np.arccos(np.clip(np.abs(A @ e), 0.0, 1.0)))


def canon(e):
    e = np.asarray(e, float)
    return -e if e[np.argmax(np.abs(e))] < 0 else e


def axial_stats(A):
    """Sign-agnostic concentration of unit axes: principal eigenvector of mean(a a^T)."""
    Tm = A.T @ A / max(len(A), 1)
    w, V = np.linalg.eigh(Tm)
    e1 = canon(V[:, -1])
    dev = axial_dev(A, e1)
    return dict(lambda1=float(w[-1]), lambdas=w[::-1].tolist(), axis=e1, median_dev=float(np.median(dev)),
                p90_dev=float(np.percentile(dev, 90)), frac_within_15=float(np.mean(dev < 15)))


def axial_modes(A, tol=15.0, max_modes=3, min_frac=0.05, seed=0):
    """Greedy search for up to 3 axis clusters (handles >1 pseudo-symmetry axis)."""
    rng = np.random.default_rng(seed)
    rem = np.ones(len(A), bool)
    modes = []
    ctol = np.cos(np.radians(tol))
    for _ in range(max_modes):
        idx = np.nonzero(rem)[0]
        if len(idx) < max(20, min_frac * len(A)):
            break
        cand = A[rng.choice(idx, min(600, len(idx)), replace=False)]
        dat = A[rng.choice(idx, min(6000, len(idx)), replace=False)]
        cnt = (np.abs(cand @ dat.T) > ctol).sum(1)
        e = cand[np.argmax(cnt)]
        for _ in range(4):
            mem = rem & (axial_dev(A, e) < tol)
            if mem.sum() < 5:
                break
            e = axial_stats(A[mem])["axis"]
        mem = rem & (axial_dev(A, e) < tol)
        if mem.sum() < min_frac * len(A):
            break
        modes.append(dict(axis=canon(e), frac=float(mem.mean()), n=int(mem.sum()),
                          median_dev=float(np.median(axial_dev(A[mem], e)))))
        rem &= ~mem
    return modes


def fmt_v(v):
    return "(" + ", ".join(f"{x:+.3f}" for x in v) + ")"


# ------------------------------- stats helpers -------------------------------
def cohen_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return float((a.mean() - b.mean()) / sp) if sp > 0 else float("nan")


def ks_2samp(a, b):
    a, b = np.sort(np.asarray(a, float)), np.sort(np.asarray(b, float))
    if len(a) == 0 or len(b) == 0:
        return float("nan"), float("nan")
    allv = np.concatenate([a, b])
    D = float(np.max(np.abs(np.searchsorted(a, allv, "right") / len(a) - np.searchsorted(b, allv, "right") / len(b))))
    en = np.sqrt(len(a) * len(b) / (len(a) + len(b)))
    lam = (en + 0.12 + 0.11 / en) * D
    if lam < 0.2:
        return D, 1.0
    j = np.arange(1, 101)
    p = 2.0 * np.sum((-1.0) ** (j - 1) * np.exp(-2.0 * j ** 2 * lam ** 2))
    return D, float(min(max(p, 0.0), 1.0))


def binom_z(k, n, p0):
    return float((k - n * p0) / np.sqrt(n * p0 * (1 - p0))) if n > 0 and 0 < p0 < 1 else float("nan")


def auc_score(s, y):
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s))
    ranks[order] = np.arange(1, len(s) + 1)
    n1, n0 = y.sum(), (~y).sum()
    return float((ranks[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")


def lda_cv_auc(Z, y, seed=0):
    """2-fold cross-validated AUC of a linear discriminant (numpy only)."""
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(Z))
    halves = (perm[: len(Z) // 2], perm[len(Z) // 2:])
    aucs = []
    for tr, te in (halves, halves[::-1]):
        Zt, yt = Z[tr], y[tr]
        m1, m0 = Zt[yt].mean(0), Zt[~yt].mean(0)
        Sw = np.cov(np.vstack([Zt[yt] - m1, Zt[~yt] - m0]).T) + 1e-6 * np.eye(Z.shape[1])
        w = np.linalg.solve(Sw, m1 - m0)
        aucs.append(auc_score(Z[te] @ w, y[te]))
    return float(np.mean(aucs))


def kmeans_np(X, k, iters=100, seed=0):
    rng = np.random.default_rng(seed)
    S = X[rng.choice(len(X), min(len(X), 50000), replace=False)]
    C = [S[rng.integers(len(S))]]
    d2 = ((S - C[0]) ** 2).sum(1)
    for _ in range(1, k):
        C.append(S[rng.choice(len(S), p=d2 / d2.sum())])
        d2 = np.minimum(d2, ((S - C[-1]) ** 2).sum(1))
    C = np.array(C)
    for _ in range(iters):
        lab = np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), 1)
        newC = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(k)])
        if np.allclose(newC, C):
            break
        C = newC
    sizes = np.bincount(lab, minlength=k)
    order = np.argsort(-sizes)  # relabel: 0 = largest cluster
    remap = np.empty(k, int)
    remap[order] = np.arange(k)
    return remap[lab], C[order]


def grid_islands(xy, nbins=120):
    """Connected 'islands' of a 2D embedding: occupied cells of a coarse histogram, 8-connectivity."""
    lo, hi = xy.min(0), xy.max(0)
    span = np.where(hi - lo > 0, hi - lo, 1.0)
    ij = np.clip(((xy - lo) / span * nbins).astype(int), 0, nbins - 1)
    H = np.zeros((nbins, nbins), int)
    np.add.at(H, (ij[:, 0], ij[:, 1]), 1)
    occ = H >= max(3, int(2e-5 * len(xy)))
    lab = -np.ones((nbins, nbins), int)
    ncomp = 0
    for i0, j0 in zip(*np.nonzero(occ)):
        if lab[i0, j0] >= 0:
            continue
        lab[i0, j0] = ncomp
        stack = [(i0, j0)]
        while stack:
            a, b = stack.pop()
            for da in (-1, 0, 1):
                for db in (-1, 0, 1):
                    na, nb = a + da, b + db
                    if 0 <= na < nbins and 0 <= nb < nbins and occ[na, nb] and lab[na, nb] < 0:
                        lab[na, nb] = ncomp
                        stack.append((na, nb))
        ncomp += 1
    pl = lab[ij[:, 0], ij[:, 1]]
    sizes = np.bincount(pl[pl >= 0], minlength=ncomp)
    order = np.argsort(-sizes)
    remap = np.empty(ncomp, int)
    remap[order] = np.arange(ncomp)
    return np.where(pl >= 0, remap[np.maximum(pl, 0)], -1)


def region_table(labels, flip, valid, p0, names=None, min_frac=0.0):
    rows = []
    n_all = valid.sum()
    for L in np.unique(labels[valid]):
        m = valid & (labels == L)
        n, k = int(m.sum()), int(flip[m].sum())
        if n < max(1, min_frac * n_all):
            continue
        f = k / n
        rows.append(dict(region=(names[L] if names else int(L)), n=n, share=n / n_all, n_flip=k, flip_frac=f,
                         enrich=f / p0 if p0 > 0 else float("nan"), z=binom_z(k, n, p0)))
    return rows


def print_table(rows, cols, fmts):
    log("  " + "  ".join(f"{c:>12s}" for c in cols))
    for r in rows:
        log("  " + "  ".join(format(r[c], f) if not isinstance(r[c], str) else f"{r[c]:>12s}" for c, f in zip(cols, fmts)))


def write_csv(rows, path):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w") as f:
        f.write(",".join(keys) + "\n")
        for r in rows:
            f.write(",".join(str(r[k]) for k in keys) + "\n")
    log(f"  [csv] {path}")


# =========================================================================================
def run():
    t0 = time.time()
    os.makedirs(OUT_DIR, exist_ok=True)
    S = {"params": dict(J486_CS=J486_CS, J487_CS=J487_CS, IND_PKL=IND_PKL, THRESH_DEG=THRESH_DEG,
                        REMOVE_GLOBAL_OFFSET=REMOVE_GLOBAL_OFFSET)}

    # ---------------------------------------------------------------- 1. load + match
    section("1. Load .cs files and match particles")
    cs486, cs487 = load_cs(J486_CS), load_cs(J487_CS)
    N486, N487 = len(cs486), len(cs487)
    log(f"J486: {N486:,} rows  ({J486_CS})")
    log(f"J487: {N487:,} rows  ({J487_CS})")
    for nm, cs in (("J486", cs486), ("J487", cs487)):
        miss = [f for f in ("uid", "alignments3D/pose", "alignments3D/shift") if f not in cs.dtype.names]
        if miss:
            raise KeyError(f"{nm} .cs lacks {miss}. If the poses live in a passthrough file "
                           f"(e.g. J486_passthrough_particles_0.cs), export the job's particles with alignments "
                           f"(or merge the passthrough by uid) and point this cell at that file.")
    u486, u487 = field(cs486, "uid").astype(np.uint64), field(cs487, "uid").astype(np.uint64)
    for nm, u in (("J486", u486), ("J487", u487)):
        nd = len(u) - len(np.unique(u))
        if nd:
            log(f"WARNING: {nm} has {nd} duplicated uids; matching keeps one row per uid.")

    def match(key487, key486):
        order = np.argsort(key486, kind="stable")
        s = key486[order]
        pos = np.clip(np.searchsorted(s, key487), 0, len(s) - 1)
        hit = s[pos] == key487
        out = np.full(len(key487), -1, np.int64)
        out[hit] = order[pos[hit]]
        return out

    m487 = match(u487, u486)
    how = "uid"
    if (m487 >= 0).mean() < 0.5:
        bp6, bi6, bp7, bi7 = (field(cs486, "blob/path"), field(cs486, "blob/idx"),
                              field(cs487, "blob/path"), field(cs487, "blob/idx"))
        if all(x is not None for x in (bp6, bi6, bp7, bi7)):
            log(f"Only {(m487 >= 0).mean():.1%} matched by uid -> matching by (blob file name, blob/idx) instead.")

            def bkey(p, i):
                p = np.char.rpartition(np.asarray(p).astype("S"), b"/")[:, 2]
                return np.char.add(np.char.add(p, b"#"), np.asarray(i).astype("S12"))

            m2 = match(bkey(bp7, bi7), bkey(bp6, bi6))
            if (m2 >= 0).sum() > (m487 >= 0).sum():
                m487, how = m2, "blob/path+blob/idx"
    matched = m487 >= 0
    nm_ = int(matched.sum())
    log(f"Matched by {how}: {nm_:,} of {N487:,} J487 rows ({nm_ / N487:.2%}); "
        f"J487-only: {N487 - nm_:,}; J486-only: {N486 - len(np.unique(m487[matched])):,}")
    if nm_ == 0:
        raise RuntimeError("No particles matched between the two files.")
    same_order = bool(nm_ == N487 and np.array_equal(m487, np.arange(N487)))
    log(f"Row order identical between files: {same_order}")
    S["match"] = dict(N486=N486, N487=N487, matched=nm_, key=how, j487_only=N487 - nm_)

    # ---------------------------------------------------------------- 2. rotations
    section("2. Relative rotations")
    pose486 = field(cs486, "alignments3D/pose").astype(np.float64)
    pose487 = field(cs487, "alignments3D/pose").astype(np.float64)
    sh486 = field(cs486, "alignments3D/shift").astype(np.float64)
    sh487 = field(cs487, "alignments3D/shift").astype(np.float64)
    Rcs486_all, Rcs487 = expmap(pose486), expmap(pose487)
    rot486_all, rot487 = T(Rcs486_all), T(Rcs487)  # cryoDRGN convention: rot = expmap(pose)^T
    D486 = int(field(cs486, "blob/shape")[0][0]) if "blob/shape" in cs486.dtype.names else None
    D487 = int(field(cs487, "blob/shape")[0][0]) if "blob/shape" in cs487.dtype.names else None
    psz = None
    for cs in (cs487, cs486):
        for f in ("alignments3D/psize_A", "blob/psize_A"):
            if psz is None and f in cs.dtype.names:
                psz = float(np.asarray(cs[f])[0])
    log(f"Box (blob/shape): J486 {D486}, J487 {D487}; pixel size {psz} A")
    log("Convention: Rcs = expmap(alignments3D/pose). cryoDRGN (parse_pose_csparc) uses rot = Rcs^T and samples the")
    log("  volume at (image coords) @ rot, i.e. x_image = rot . x_map: rot maps MAP (particle) -> IMAGE coordinates.")
    log("  Map/body-frame relative rotation  Rb = rot486^T rot487  (= Rcs486 Rcs487^T)")
    log("  Image/lab-frame relative rotation Rl = rot487 rot486^T  (= Rcs487^T Rcs486)")
    log("  A flip of the particle about its own 2-fold u gives rot487 = rot486 C_u, so Rb = C_u has the SAME axis for")
    log("  every particle, while Rl's axis (= rot486 u) moves with the viewing direction.")
    log("  Caution: the frame names come from cryoDRGN's convention, which reproduces cryoSPARC maps. If you plug raw")
    log("  Rcs matrices into the textbook formulas, the two names swap.")

    if POSE_PKL and opt_path(POSE_PKL):
        try:
            pp = load_pkl(POSE_PKL)
            rot_p = np.asarray(pp[0] if isinstance(pp, (tuple, list)) else pp)
            if len(rot_p) == N487:
                smp = np.random.default_rng(1).choice(N487, min(20000, N487), replace=False)
                dif = np.abs(rot_p[smp] - rot487[smp]).reshape(len(smp), -1).max(1)
                n_nan = int(np.sum(~np.isfinite(dif)))  # cryoDRGN's expmap gives NaN for an exactly-zero pose
                err = float(np.nanmax(dif)) if n_nan < len(smp) else float("nan")
                ok = bool(np.isfinite(err) and err < 1e-3)
                log(f"pose.pkl check: max |rot(pose.pkl) - expmap(J487 pose)^T| = {err:.2e} on {len(smp) - n_nan} rows"
                    + (f" ({n_nan} NaN rows in pose.pkl ignored)" if n_nan else "") + " -> "
                    + ("OK: pose.pkl row i == J487 .cs row i, convention confirmed." if ok else "MISMATCH: check row order / convention!"))
                S["pose_pkl_check"] = dict(max_err=err, ok=ok, nan_rows=n_nan)
            else:
                log(f"pose.pkl has {len(rot_p)} rows, J487 has {N487}: not compared.")
        except Exception as e:
            log(f"pose.pkl check skipped ({e})")

    mi = np.nonzero(matched)[0]  # J487 rows that have a J486 partner
    j6 = m487[mi]
    rot486 = rot486_all[j6]
    r7 = rot487[mi]
    Rb = np.einsum("nji,njk->nik", rot486, r7)  # rot486^T rot487
    ang_raw, _ = angle_axis(Rb)
    log(f"Raw relative angle: median {np.median(ang_raw):.2f} deg; >{THRESH_DEG:g} deg: {np.mean(ang_raw > THRESH_DEG):.2%}")

    # global frame offset (e.g. J487 refined against a reference in a different orientation)
    start = ang_raw < 90 if np.sum(ang_raw < 90) >= np.sum(ang_raw >= 90) else np.ones(len(ang_raw), bool)
    if np.sum(ang_raw < 90) < np.sum(ang_raw >= 90):
        log("WARNING: most particles differ by >90 deg; the 'reference' register below is the majority group.")
    sel = start
    G = np.eye(3)
    for _ in range(6):
        G = proj_so3(Rb[sel].mean(0))
        a_, _ = angle_axis(np.einsum("nij,kj->nik", Rb, G))
        sel = a_ < 30
        if sel.sum() < 10:
            break
    gang, gax = angle_axis(G[None])
    gang = float(gang[0])
    log(f"Global map-frame offset between jobs (robust mean of Rb over agreeing particles): {gang:.2f} deg "
        f"about {fmt_v(gax[0])}")
    applied = False
    if REMOVE_GLOBAL_OFFSET and gang > 1.0:
        Rb = np.einsum("nij,kj->nik", Rb, G)  # Rb G^T  ==  J487 rotations re-expressed in J486's map frame
        r7 = np.einsum("nij,kj->nik", r7, G)
        applied = True
        log(f"  -> removed (REMOVE_GLOBAL_OFFSET=True). Axes below are in J486's map frame.")
    elif gang > 1.0:
        log("  -> NOT removed (REMOVE_GLOBAL_OFFSET=False); angles include this offset.")
    S["global_offset"] = dict(angle_deg=gang, axis=gax[0].tolist(), removed=applied)
    Rl = np.einsum("nij,nkj->nik", r7, rot486)  # rot487 rot486^T
    ang, ax_b = angle_axis(Rb)
    ang_l, ax_l = angle_axis(Rl)
    log(f"Angle check (body vs lab formula, should be identical): max diff {np.abs(ang - ang_l).max():.2e} deg")

    # ---------------------------------------------------------------- 3. classify
    section(f"3. Flip classification (flipped = relative angle > {THRESH_DEG:g} deg)")
    flip_m = ang > THRESH_DEG
    nf = int(flip_m.sum())

    # shifts: which reference do J486's (passthrough) shifts have? Decide on the agreeing particles.
    for cs_, nm_cs in ((cs486, "J486"), (cs487, "J487")):
        if "alignments3D/psize_A" in cs_.dtype.names:
            _p = float(np.asarray(cs_["alignments3D/psize_A"])[0])
            if psz and abs(_p - psz) > 1e-3 * psz:
                log(f"WARNING: {nm_cs} alignments3D/psize_A = {_p} differs from {psz}: shift units differ between jobs.")
    cands = {"as_is": sh486, "zero": np.zeros_like(sh486), "frac": sh486 - np.round(sh486)}
    agree = ang < min(10.0, THRESH_DEG)
    if agree.sum() < 100:
        agree = ~flip_m
    med = {k: float(np.median(np.linalg.norm(sh487[mi][agree] - v[j6][agree], axis=1))) if agree.any() else float("nan")
           for k, v in cands.items()}
    mode = str(J486_SHIFT_MODE).strip().lower()
    if mode not in cands:
        best_k = min(med, key=lambda k: med[k] if np.isfinite(med[k]) else np.inf)
        mode = best_k if med[best_k] < 0.8 * med["as_is"] else "as_is"
        how_mode = "auto"
    else:
        how_mode = "forced"
    sh486_eff = cands[mode]
    log(f"J486 shift reference ({how_mode}): '{mode}'. Median |shift J487 - J486| over {int(agree.sum()):,} agreeing "
        f"particles if J486 shifts are taken as_is {med['as_is']:.2f} px / as zero (exact recentring) {med['zero']:.2f} px"
        f" / as fractional part (integer recentring) {med['frac']:.2f} px.")
    if mode != "as_is":
        log("  -> J486's alignments3D shifts do NOT refer to the J486 particle images (RBMC recentring; alignments3D")
        log("     passed through from J485). Shift comparisons and the J486 pose file below use the corrected reference.")
    S["j486_shift_model"] = dict(mode=mode, how=how_mode, median_px=med)
    dsh = sh487[mi] - sh486_eff[j6]
    dsh_n = np.linalg.norm(dsh, axis=1)
    edges = [0, 2, 5, 10, 20, 45, 90, 135, THRESH_DEG, 170, 175, 180.0001]
    edges = sorted(set(edges))
    cnt = np.histogram(ang, bins=edges)[0]
    log("Relative-angle distribution (matched particles):")
    for a, b, c in zip(edges[:-1], edges[1:], cnt):
        log(f"  {a:6.1f} - {min(b, 180):6.1f} deg : {c:9,d}  ({c / len(ang):7.3%})")
    ind = np.asarray(load_pkl(IND_PKL)).astype(np.int64).ravel()
    if ind.max() >= N487 or ind.min() < 0:
        raise ValueError(f"IND_PKL indices out of range for J487 ({ind.min()}..{ind.max()} vs {N487})")
    angle_g = np.full(N487, np.nan)
    angle_g[mi] = ang
    flipped_g = np.zeros(N487, bool)
    flipped_g[mi[flip_m]] = True
    unflipped_g = np.zeros(N487, bool)
    unflipped_g[mi[~flip_m]] = True
    in_ind = np.zeros(N487, bool)
    in_ind[ind] = True
    nf_ind, nm_ind = int((flipped_g & in_ind).sum()), int((matched & in_ind).sum())
    p_all = nf / len(ang)
    p_ind = nf_ind / max(nm_ind, 1)
    log(f"Flipped: {nf:,} / {len(ang):,} matched = {p_all:.3%}  (1 in {1 / p_all:.1f})" if nf else "Flipped: 0")
    log(f"Flipped in IND subset ({len(ind):,} particles, {nm_ind:,} matched): {nf_ind:,} = {p_ind:.3%}")
    log(f"Flipped outside IND: {nf - nf_ind:,} of {len(ang) - nm_ind:,} = "
        f"{(nf - nf_ind) / max(len(ang) - nm_ind, 1):.3%}")
    un = ang[~flip_m]
    log(f"Non-flipped group: median {np.median(un):.2f} deg, 95th pct {np.percentile(un, 95):.2f} deg, "
        f"99th pct {np.percentile(un, 99):.2f} deg")
    if nf:
        log(f"Flipped group:     median {np.median(ang[flip_m]):.2f} deg, 5th-95th pct "
            f"{np.percentile(ang[flip_m], 5):.2f}-{np.percentile(ang[flip_m], 95):.2f} deg")
    log(f"Shift change |d shift| (px): non-flipped median {np.median(dsh_n[~flip_m]):.2f}, 95th {np.percentile(dsh_n[~flip_m], 95):.2f}"
        + (f"; flipped median {np.median(dsh_n[flip_m]):.2f}, 95th {np.percentile(dsh_n[flip_m], 95):.2f}" if nf else ""))
    if psz:
        log(f"  (x {psz:.4f} A/px: non-flipped median {np.median(dsh_n[~flip_m]) * psz:.2f} A"
            + (f", flipped median {np.median(dsh_n[flip_m]) * psz:.2f} A)" if nf else ")"))
    S["classify"] = dict(n_matched=len(ang), n_flip=nf, frac_flip=p_all, n_flip_ind=nf_ind, n_matched_ind=nm_ind,
                         frac_flip_ind=p_ind, unflipped_median_deg=float(np.median(un)),
                         unflipped_p95_deg=float(np.percentile(un, 95)),
                         flipped_median_deg=float(np.median(ang[flip_m])) if nf else None,
                         dshift_median_px_unflipped=float(np.median(dsh_n[~flip_m])),
                         dshift_median_px_flipped=float(np.median(dsh_n[flip_m])) if nf else None)

    fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
    axs[0].hist(ang, bins=np.arange(0, 181, 1), color=C_UNF, edgecolor="none")
    axs[0].set_yscale("log")
    axs[0].axvline(THRESH_DEG, color=C_MUTED, ls="--", lw=1)
    axs[0].set_xlabel("relative rotation J486 -> J487 (deg)", fontsize=9)
    axs[0].set_ylabel("particles (log)", fontsize=9)
    axs[0].set_title(f"Relative pose angle: {p_all:.1%} above {THRESH_DEG:g} deg", fontsize=10, loc="left")
    axs[1].hist(dsh_n[~flip_m], bins=np.linspace(0, np.percentile(dsh_n, 99.5) + 1e-6, 80), color=C_UNF,
                alpha=0.8, density=True, label="non-flipped")
    if nf:
        axs[1].hist(dsh_n[flip_m], bins=np.linspace(0, np.percentile(dsh_n, 99.5) + 1e-6, 80), color=C_FLIP,
                    alpha=0.8, density=True, label="flipped")
    axs[1].set_xlabel("|shift J487 - shift J486| (px)", fontsize=9)
    axs[1].set_title("Shift change", fontsize=10, loc="left")
    axs[1].legend(frameon=False, fontsize=8)
    for a in axs:
        style_ax(a)
    savefig(fig, "relative_angle_hist.png")

    # ---------------------------------------------------------------- 4. axis consistency
    section("4. Is there ONE flip axis? (axis spread for flipped particles)")
    verdict_frame, body_axis, lab_axis = None, None, None
    stb = stl = None
    modes_b = modes_l = []
    if nf >= 10:
        Ab, Al = ax_b[flip_m], ax_l[flip_m]
        stb, stl = axial_stats(Ab), axial_stats(Al)
        modes_b, modes_l = axial_modes(Ab), axial_modes(Al)
        nrand = 1 - np.cos(np.radians(15))
        log(f"(Random axes would give lambda1 ~ 0.33, median deviation ~ 60 deg, ~{nrand:.1%} within 15 deg.)")
        for nm, st, md in (("MAP/body frame  (Rb)", stb, modes_b), ("IMAGE/lab frame (Rl)", stl, modes_l)):
            log(f"{nm}: lambda1 {st['lambda1']:.3f} (eig {st['lambdas'][0]:.3f}/{st['lambdas'][1]:.3f}/{st['lambdas'][2]:.3f}); "
                f"principal axis {fmt_v(st['axis'])}; median dev {st['median_dev']:.2f} deg; "
                f"90th pct {st['p90_dev']:.2f} deg; within 15 deg {st['frac_within_15']:.1%}")
            for i, m in enumerate(md):
                log(f"    axis cluster {i + 1}: {fmt_v(m['axis'])}  {m['frac']:.1%} of flips, median dev {m['median_dev']:.2f} deg")
        fb = modes_b[0]["frac"] if modes_b else 0.0
        fl = modes_l[0]["frac"] if modes_l else 0.0
        body_axis = modes_b[0]["axis"] if modes_b else stb["axis"]
        lab_axis = modes_l[0]["axis"] if modes_l else stl["axis"]
        if fb >= 0.5 and fl < 0.5:
            verdict_frame = "body"
        elif fl >= 0.5 and fb < 0.5:
            verdict_frame = "lab"
        elif fl >= 0.5 and fb >= 0.5:
            verdict_frame = "both"
        else:
            verdict_frame = "none"
        log(f"-> tight axis cluster in: {dict(body='the MAP (particle) frame', lab='the IMAGE (lab) frame', both='BOTH frames', none='neither frame')[verdict_frame]}")
        if verdict_frame in ("body", "both"):
            e = body_axis
            angs = np.degrees(np.arccos(np.clip(np.abs(e), 0, 1)))
            log(f"   Map-frame flip axis u = {fmt_v(e)}; angle to map x/y/z: {angs[0]:.1f}/{angs[1]:.1f}/{angs[2]:.1f} deg")
            sel_mode = flip_m.copy()
            sel_mode[flip_m] = axial_dev(Ab, e) < 15
            log(f"   rotation angle about u (flips within 15 deg of u): median {np.median(ang[sel_mode]):.2f} deg "
                "(a perfect 2-fold gives 180)")
        if verdict_frame in ("lab", "both"):
            log(f"   Image-frame flip axis {fmt_v(lab_axis)}; angle to beam (z): "
                f"{np.degrees(np.arccos(min(1, abs(lab_axis[2])))):.1f} deg (~0 = in-plane 180 deg rotation)")
        S["axis"] = dict(frame=verdict_frame, body=dict(stb, axis=stb["axis"].tolist()),
                         lab=dict(stl, axis=stl["axis"].tolist()),
                         body_modes=[dict(m, axis=m["axis"].tolist()) for m in modes_b],
                         lab_modes=[dict(m, axis=m["axis"].tolist()) for m in modes_l],
                         body_axis=body_axis.tolist(), lab_axis=lab_axis.tolist())

        # equal-area plot of the axes, both frames
        rng = np.random.default_rng(0)
        sub = rng.choice(nf, min(nf, 20000), replace=False)
        fig, axs = plt.subplots(1, 2, figsize=(9.5, 4.6), layout="constrained")
        for a, A, st, title in ((axs[0], Ab, stb, "map (body) frame"), (axs[1], Al, stl, "image (lab) frame")):
            P = A[sub] * np.where(A[sub, 2:3] < 0, -1, 1)  # fold to upper hemisphere (axis sign is arbitrary)
            r = np.sqrt(np.clip(2 * (1 - P[:, 2]), 0, None)) / np.sqrt(2)
            phi = np.arctan2(P[:, 1], P[:, 0])
            a.scatter(r * np.cos(phi), r * np.sin(phi), s=2, alpha=0.25, color=C_FLIP, edgecolors="none")
            a.add_patch(plt.Circle((0, 0), 1, fill=False, color=C_MUTED, lw=0.8))
            a.set_xlim(-1.05, 1.05)
            a.set_ylim(-1.05, 1.05)
            a.set_aspect("equal")
            a.axis("off")
            a.set_title(f"Flip axes, {title}\nlambda1 = {st['lambda1']:.2f}, median dev {st['median_dev']:.1f} deg",
                        fontsize=9, loc="left")
            a.text(1.0, 0, "x", color=C_MUTED, fontsize=8)
            a.text(0, 1.0, "y", color=C_MUTED, fontsize=8)
            a.text(0, 0, "+", color=C_MUTED, fontsize=10, ha="center", va="center")
        axs[0].text(-1.05, -1.12, "equal-area view of the upper hemisphere; centre = +z", fontsize=7, color=C_MUTED)
        savefig(fig, "flip_axes.png")
    else:
        log("Fewer than 10 flipped particles: axis test skipped.")

    # 4b. where does the 2-fold axis sit, and does the flip depend on viewing direction?
    if verdict_frame in ("body", "both") and nf >= 50:
        e = np.asarray(body_axis)
        C = 2 * np.outer(e, e) - np.eye(3)
        # shift model: dt = P_xy rot486 (I - C) p, p = axis position relative to box centre (px).
        # The sign of p depends on cryoSPARC's shift convention; its size does not.
        fsel = np.nonzero(flip_m)[0]
        A = (rot486[fsel] @ (np.eye(3) - C))[:, :2, :].reshape(-1, 3)
        y = dsh[fsel].reshape(-1)
        p, *_ = np.linalg.lstsq(A, y, rcond=None)
        p -= e * (p @ e)
        res = y - A @ p
        r2 = 1 - res.var() / y.var() if y.var() > 0 else float("nan")
        log(f"2-fold axis position from the shift changes of flipped particles: {np.linalg.norm(p):.2f} px"
            + (f" = {np.linalg.norm(p) * psz:.1f} A" if psz else "") + f" from the box centre (fit R^2 = {r2:.2f}; "
            "sign convention-dependent)")
        log("   R^2 near 0 means the shift changes are noise, i.e. the axis passes ~through the box centre.")
        S["axis_offset"] = dict(px=float(np.linalg.norm(p)), vec_px=p.tolist(), r2=float(r2))

        beam_b = rot486_all[:, 2, :]  # beam (image z) in map coords = rot^T e_z = 3rd row of rot
        cosv = np.abs(beam_b[j6] @ e)
        bins = np.linspace(0, 1, 11)
        bi = np.clip(np.digitize(cosv, bins) - 1, 0, 9)
        rows = []
        for b in range(10):
            m = bi == b
            rows.append(dict(cos_lo=bins[b], cos_hi=bins[b + 1], n=int(m.sum()),
                             flip_frac=float(flip_m[m].mean()) if m.any() else float("nan")))
        log("Flip fraction vs viewing direction (|cos| of angle between beam and the 2-fold; 1 = looking down the axis):")
        for r in rows:
            log(f"  |cos| {r['cos_lo']:.1f}-{r['cos_hi']:.1f}: n {r['n']:8,d}  flipped {r['flip_frac']:.2%}")
        ff = np.array([r["flip_frac"] for r in rows])
        log(f"  range of flip fraction across view bins: {np.nanmin(ff):.2%} - {np.nanmax(ff):.2%} (overall {p_all:.2%})")
        S["view_dependence"] = rows
        write_csv(rows, os.path.join(OUT_DIR, "flip_vs_view.csv"))
        fig, a = plt.subplots(figsize=(6, 3.4))
        a.bar((bins[:-1] + bins[1:]) / 2, ff, width=0.09, color=C_FLIP, edgecolor="none")
        a.axhline(p_all, color=C_MUTED, ls="--", lw=1)
        a.text(0.01, p_all, f" overall {p_all:.1%}", va="bottom", fontsize=8, color=C_MUTED)
        a.set_xlabel("|cos(beam, 2-fold axis)|   (0 = side view of the axis, 1 = down the axis)", fontsize=9)
        a.set_ylabel("fraction flipped", fontsize=9)
        a.set_title("Flip fraction vs viewing direction", fontsize=10, loc="left")
        style_ax(a)
        savefig(fig, "flip_vs_view.png")

    # ---------------------------------------------------------------- 5. outputs in cryoDRGN numbering
    section("5. Index files (global index = J487 row = cryoDRGN particle index)")
    fl_g = np.sort(np.nonzero(flipped_g)[0]).astype(np.int64)
    un_g = np.sort(np.nonzero(unflipped_g)[0]).astype(np.int64)
    outs = {"flipped_global_ind.pkl": fl_g, "unflipped_global_ind.pkl": un_g,
            "flipped_global_ind_in_ind.pkl": np.intersect1d(fl_g, ind).astype(np.int64),
            "unflipped_global_ind_in_ind.pkl": np.intersect1d(un_g, ind).astype(np.int64)}
    for k, v in outs.items():
        save_pkl(v, os.path.join(OUT_DIR, k))
        log(f"  [pkl] {os.path.join(OUT_DIR, k)}  ({len(v):,} indices)")
    flip_f = flipped_g[ind]
    matched_f = matched[ind]
    np.save(os.path.join(OUT_DIR, "flip_filtered.npy"), flip_f)
    np.save(os.path.join(OUT_DIR, "matched_filtered.npy"), matched_f)
    log(f"  [npy] flip_filtered.npy / matched_filtered.npy: one bool per particle of the --ind run, in IND_PKL order "
        f"({len(flip_f):,}; {int(flip_f.sum()):,} flipped, {int((~matched_f).sum()):,} unmatched)")
    np.savez_compressed(os.path.join(OUT_DIR, "per_particle_global.npz"), angle_deg=angle_g.astype(np.float32),
                        j486_row=m487, flipped=flipped_g)
    log(f"  [npz] per_particle_global.npz (relative angle, J486 row, flipped flag per J487 row)")
    if WRITE_J486_POSE_PKL:
        rot_o = rot487.astype(np.float32).copy()
        rot_o[mi] = rot486_all[j6].astype(np.float32)
        Dn = D486 or D487
        trans_o = (sh487 / (D487 or Dn)).astype(np.float32)
        trans_o[mi] = (sh486_eff[j6] / Dn).astype(np.float32)  # J486 shifts in the reference of the particle stack
        pth = os.path.join(OUT_DIR, "pose_J486_in_J487_order.pkl")
        save_pkl((rot_o, trans_o), pth)
        log(f"  [pkl] {pth}: cryoDRGN pose file with J486 poses in J487/cryoDRGN order "
            f"({N487 - nm_:,} unmatched rows keep J487 poses; J486 shifts taken '{mode}'). Use it to back-project the "
            "flipped subset both ways.")
        if applied:
            log(f"        (J486 poses are in J486's map frame, {gang:.1f} deg from J487's: the two back-projections will be "
                "rotated relative to each other by that amount.)")

    # ---------------------------------------------------------------- 6. relation to heterogeneity
    section("6. Do the flips line up with heterogeneity / metadata?")
    assoc = []
    valid_f = matched_f
    p0f = flip_f[valid_f].mean() if valid_f.any() else 0.0

    # 6a. UMAP
    if opt_path(UMAP_PKL):
        U = np.asarray(load_pkl(UMAP_PKL), float)
        if U.ndim == 2 and U.shape[1] >= 2 and len(U) == len(ind):
            U = U[:, :2]
            log(f"UMAP: {UMAP_PKL} ({len(U):,} points, filtered order)")
            isl = grid_islands(U)
            rows_i = region_table(isl, flip_f, valid_f & (isl >= 0), p0f, min_frac=0.003)
            log(f"Connected islands (coarse 2D histogram, components holding >= 0.3% of points; overall {p0f:.2%} flipped):")
            print_table(rows_i, ["region", "n", "share", "flip_frac", "enrich", "z"], ["12d", "12,d", "12.2%", "12.2%", "12.2f", "12.1f"])
            km, cen = kmeans_np(U, int(UMAP_KMEANS_K))
            rows_k = region_table(km, flip_f, valid_f, p0f)
            log(f"k-means (k={UMAP_KMEANS_K}) on UMAP coordinates:")
            print_table(rows_k, ["region", "n", "share", "flip_frac", "enrich", "z"], ["12d", "12,d", "12.2%", "12.2%", "12.2f", "12.1f"])
            write_csv([dict(r, kind="island") for r in rows_i] + [dict(r, kind="kmeans") for r in rows_k],
                      os.path.join(OUT_DIR, "umap_region_flip.csv"))
            best = max(rows_i + rows_k, key=lambda r: abs(np.log(max(r["enrich"], 1e-9))) if r["share"] >= 0.01 else 0)
            S["umap"] = dict(islands=rows_i, kmeans=rows_k)
            if best["share"] >= 0.01 and (best["enrich"] >= 1.5 or best["enrich"] <= 1 / 1.5) and abs(best["z"]) > 5:
                assoc.append(f"UMAP region with {best['share']:.1%} of particles has {best['flip_frac']:.1%} flipped "
                             f"({best['enrich']:.2f}x overall, z = {best['z']:.0f})")
            # plots
            rng = np.random.default_rng(0)
            fig, axs = plt.subplots(1, 3, figsize=(16, 4.8), layout="constrained")
            vu = np.nonzero(valid_f & ~flip_f)[0]
            vf = np.nonzero(valid_f & flip_f)[0]
            su = rng.choice(vu, min(len(vu), 120000), replace=False)
            sf = rng.choice(vf, min(len(vf), 40000), replace=False) if len(vf) else vf
            axs[0].scatter(U[su, 0], U[su, 1], s=0.5, color=C_UNF, alpha=0.25, edgecolors="none", label="non-flipped")
            axs[0].scatter(U[sf, 0], U[sf, 1], s=0.5, color=C_FLIP, alpha=0.5, edgecolors="none", label="flipped")
            axs[0].legend(markerscale=12, frameon=False, fontsize=8, loc="best")
            axs[0].set_title("UMAP coloured by flip", fontsize=10, loc="left")
            nb = 80
            xe = np.linspace(U[:, 0].min(), U[:, 0].max(), nb + 1)
            ye = np.linspace(U[:, 1].min(), U[:, 1].max(), nb + 1)
            Hn = np.histogram2d(U[valid_f, 0], U[valid_f, 1], bins=[xe, ye])[0]
            Hf = np.histogram2d(U[valid_f & flip_f, 0], U[valid_f & flip_f, 1], bins=[xe, ye])[0]
            frac = np.where(Hn >= 20, Hf / np.maximum(Hn, 1), np.nan)
            im = axs[1].imshow(frac.T, origin="lower", extent=[xe[0], xe[-1], ye[0], ye[-1]], aspect="auto",
                               cmap="Oranges", vmin=0, vmax=max(0.05, np.nanpercentile(frac, 99) if np.isfinite(frac).any() else 1))
            plt.colorbar(im, ax=axs[1], fraction=0.046, pad=0.02).set_label("fraction flipped", fontsize=8)
            axs[1].set_title("Fraction flipped per UMAP bin (bins with >= 20 particles)", fontsize=10, loc="left")
            sub = rng.choice(len(U), min(len(U), 120000), replace=False)
            kk = int(UMAP_KMEANS_K)
            pal = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
            cmap_k = matplotlib.colors.ListedColormap(pal[:kk]) if kk <= 8 else "tab20"
            axs[2].scatter(U[sub, 0], U[sub, 1], s=0.5, c=km[sub], cmap=cmap_k, vmin=-0.5, vmax=kk - 0.5, alpha=0.3,
                           edgecolors="none")
            for r in rows_k:
                c = cen[r["region"]]
                axs[2].text(c[0], c[1], f"k{r['region']}\n{r['flip_frac']:.1%}", fontsize=8, ha="center", va="center",
                            color=C_INK, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=C_GRID, alpha=0.85))
            axs[2].set_title(f"k-means (k={UMAP_KMEANS_K}): fraction flipped per cluster", fontsize=10, loc="left")
            for a in axs:
                style_ax(a)
                a.set_xlabel("UMAP1", fontsize=8)
                a.set_ylabel("UMAP2", fontsize=8)
            savefig(fig, "umap_flip.png")
        else:
            log(f"UMAP skipped: shape {U.shape} does not match IND length {len(ind)} (needs the filtered run's umap.pkl).")

    # 6b. latent z
    if opt_path(Z_PKL):
        Z = np.asarray(load_pkl(Z_PKL), float)
        if Z.ndim == 2 and len(Z) == len(ind) and valid_f.sum() > 100 and flip_f[valid_f].any():
            Zv, yv = Z[valid_f], flip_f[valid_f]
            ds = [cohen_d(Zv[yv, j], Zv[~yv, j]) for j in range(Z.shape[1])]
            auc = lda_cv_auc(Zv, yv)
            log(f"Latent z ({Z.shape[1]} dims): Cohen's d per dim (flipped - non-flipped): "
                + ", ".join(f"z{j}:{d:+.2f}" for j, d in enumerate(ds)))
            log(f"  linear discriminant, 2-fold cross-validated AUC = {auc:.3f} (0.5 = latent does not encode the flip)")
            S["latent"] = dict(d=ds, lda_auc=auc)
            if auc >= 0.6:
                assoc.append(f"latent z predicts flip status (cross-validated AUC {auc:.2f})")
        else:
            log(f"Z_PKL skipped (shape {Z.shape} vs IND {len(ind)}).")

    # 6c. volume PCs
    if opt_path(VOLPCA_ALL):
        Vp = np.asarray(load_pkl(VOLPCA_ALL), float)
        if Vp.ndim == 2 and len(Vp) == len(ind) and flip_f[valid_f].any():
            k = min(3, Vp.shape[1])
            rows = []
            fig, axs = plt.subplots(1, k, figsize=(4.4 * k, 3.4), layout="constrained")
            axs = np.atleast_1d(axs)
            for j in range(k):
                a_, b_ = Vp[valid_f & flip_f, j], Vp[valid_f & ~flip_f, j]
                d = cohen_d(a_, b_)
                D, p = ks_2samp(a_, b_)
                rows.append(dict(pc=f"PC{j + 1}", mean_flipped=float(a_.mean()), mean_unflipped=float(b_.mean()),
                                 cohen_d=d, ks_D=D, ks_p=p))
                bins = np.linspace(*np.percentile(Vp[valid_f, j], [0.5, 99.5]), 80)
                axs[j].hist(b_, bins=bins, density=True, color=C_UNF, alpha=0.75, label="non-flipped")
                axs[j].hist(a_, bins=bins, density=True, color=C_FLIP, alpha=0.75, label="flipped")
                axs[j].set_title(f"Volume PC{j + 1}: d = {d:+.2f}, KS D = {D:.3f}", fontsize=10, loc="left")
                axs[j].set_xlabel(f"vol PC{j + 1}", fontsize=8)
                style_ax(axs[j])
            axs[0].legend(frameon=False, fontsize=8)
            savefig(fig, "volpca_flip_hist.png")
            log(f"Volume PCs ({VOLPCA_ALL}):")
            for r in rows:
                log(f"  {r['pc']}: mean flipped {r['mean_flipped']:+.3f} vs non-flipped {r['mean_unflipped']:+.3f}; "
                    f"Cohen's d {r['cohen_d']:+.3f}; KS D {r['ks_D']:.3f} (p {r['ks_p']:.1e})")
                if abs(r["cohen_d"]) >= 0.2 or (r["ks_D"] >= 0.1 and r["ks_p"] < 1e-3):
                    assoc.append(f"volume {r['pc']} differs between flipped and non-flipped (d {r['cohen_d']:+.2f}, KS D {r['ks_D']:.2f})")
            write_csv(rows, os.path.join(OUT_DIR, "volpca_flip_stats.csv"))
            S["volpca"] = rows
        else:
            log(f"VOLPCA_ALL skipped (shape {Vp.shape} vs IND {len(ind)}).")

    # 6d. landscape states (global indices)
    if LANDSCAPE_STATES_DIR and os.path.isdir(LANDSCAPE_STATES_DIR):
        files = sorted(glob.glob(os.path.join(LANDSCAPE_STATES_DIR, "state_*_particle_ind.pkl")))
        rows = []
        for fpath in files:
            gi = np.asarray(load_pkl(fpath)).astype(np.int64).ravel()
            gi = gi[(gi >= 0) & (gi < N487)]
            frac_in = np.isin(gi, ind).mean() if len(gi) else 0
            gm = gi[matched[gi]]
            k_ = int(flipped_g[gm].sum())
            rows.append(dict(state=os.path.basename(fpath).split("_particle")[0], n=len(gm), n_flip=k_,
                             flip_frac=k_ / max(len(gm), 1), enrich=(k_ / max(len(gm), 1)) / p_ind if p_ind else float("nan"),
                             z=binom_z(k_, len(gm), p_ind), frac_in_ind=float(frac_in)))
        if rows:
            log(f"Landscape states ({LANDSCAPE_STATES_DIR}); expected flip fraction {p_ind:.2%}:")
            print_table(rows, ["state", "n", "flip_frac", "enrich", "z"], ["12s", "12,d", "12.2%", "12.2f", "12.1f"])
            if min(r["frac_in_ind"] for r in rows) < 0.99:
                log("  NOTE: some state indices are not in IND_PKL -> they may not be global indices.")
            write_csv(rows, os.path.join(OUT_DIR, "landscape_state_flip.csv"))
            S["states"] = rows
            fig, a = plt.subplots(figsize=(max(5, 0.55 * len(rows)), 3.2))
            a.bar(range(len(rows)), [r["flip_frac"] for r in rows], color=C_FLIP, edgecolor="none", width=0.7)
            a.axhline(p_ind, color=C_MUTED, ls="--", lw=1)
            a.set_xticks(range(len(rows)))
            a.set_xticklabels([r["state"].replace("state_", "") for r in rows], fontsize=8)
            a.set_ylabel("fraction flipped", fontsize=9)
            a.set_title("Flip fraction per landscape state (dashed = overall)", fontsize=10, loc="left")
            style_ax(a)
            savefig(fig, "landscape_state_flip.png")
            big = [r for r in rows if r["n"] >= 200 and (r["enrich"] >= 1.5 or r["enrich"] <= 1 / 1.5) and abs(r["z"]) > 5]
            for r in big:
                assoc.append(f"landscape {r['state']} has {r['flip_frac']:.1%} flipped ({r['enrich']:.2f}x, z = {r['z']:.0f})")
        else:
            log(f"No state_*_particle_ind.pkl in {LANDSCAPE_STATES_DIR}")

    # 6e. micrographs
    mic = field(cs487, "location/micrograph_uid")
    if mic is not None and nf:
        mu, inv = np.unique(mic[mi], return_inverse=True)
        n_m = np.bincount(inv)
        k_m = np.bincount(inv, weights=flip_m.astype(float))
        e_m = n_m * p_all
        chi = float(np.sum((k_m - e_m) ** 2 / np.maximum(e_m * (1 - p_all), 1e-12)))
        disp = chi / max(len(mu) - 1, 1)
        zero_obs = int(np.sum(k_m == 0))
        zero_exp = float(np.sum((1 - p_all) ** n_m))
        log(f"Micrographs: {len(mu):,}; flips per micrograph dispersion index {disp:.2f} (1 = random scatter, "
            f">>1 = flips cluster on some micrographs); micrographs with 0 flips: {zero_obs:,} (expected {zero_exp:,.0f})")
        S["micrographs"] = dict(n=len(mu), dispersion=disp, zero_obs=zero_obs, zero_exp=zero_exp)
        rows = [dict(micrograph_uid=int(u_), n=int(n_), n_flip=int(k_), flip_frac=float(k_ / n_))
                for u_, n_, k_ in zip(mu, n_m, k_m)]
        rows.sort(key=lambda r: -r["flip_frac"])
        write_csv(rows, os.path.join(OUT_DIR, "per_micrograph_flip.csv"))
        if disp > 1.5:
            assoc.append(f"flips cluster by micrograph (dispersion index {disp:.1f})")

    # 6f. per-particle scores / metadata
    if nf:
        cand = []
        for nm in cs487.dtype.names:
            if not nm.startswith(("alignments3D/", "ctf/", "pick_stats/", "ml_properties/")):
                continue
            if nm in ("alignments3D/pose", "alignments3D/shift", "alignments3D/psize_A"):
                continue
            dt = cs487.dtype[nm]
            if dt.shape != () or dt.kind != "f":
                continue
            cand.append(nm)
        rows = []
        for nm in cand:
            x7 = np.asarray(cs487[nm], float)[mi]
            if not np.isfinite(x7).all() or x7.std() == 0:
                continue
            r = dict(field=nm, med_flipped=float(np.median(x7[flip_m])), med_unflipped=float(np.median(x7[~flip_m])),
                     d_J487=cohen_d(x7[flip_m], x7[~flip_m]), d_J486=float("nan"))
            if nm in cs486.dtype.names:
                x6 = np.asarray(cs486[nm], float)[j6]
                if np.isfinite(x6).all() and x6.std() > 0:
                    r["d_J486"] = cohen_d(x6[flip_m], x6[~flip_m])
            rows.append(r)
        rows.sort(key=lambda r: -abs(r["d_J487"]) if np.isfinite(r["d_J487"]) else 0)
        if rows:
            log("Per-particle fields, flipped vs non-flipped (Cohen's d; |d| >= 0.2 is worth a look):")
            for r in rows[:15]:
                log(f"  {r['field']:34s} median {r['med_flipped']:.4g} vs {r['med_unflipped']:.4g}; "
                    f"d(J487) {r['d_J487']:+.2f}  d(J486) {r['d_J486']:+.2f}")
            write_csv(rows, os.path.join(OUT_DIR, "metadata_flip_stats.csv"))
            S["metadata"] = rows
        for cs, nmj, rows_idx in ((cs487, "J487", mi), (cs486, "J486", j6)):
            spl = field(cs, "alignments3D/split")
            if spl is not None:
                s_ = spl[rows_idx]
                parts = [f"split {v}: {flip_m[s_ == v].mean():.2%} of {int((s_ == v).sum()):,}" for v in np.unique(s_)[:4]]
                log(f"  {nmj} half-set (alignments3D/split) flip fractions: " + "; ".join(parts))

    # ---------------------------------------------------------------- 7. verdict
    section("7. VERDICT")
    log(f"Particles compared: {len(ang):,} (J487 rows with a J486 partner, matched by {how}).")
    if gang > 1.0:
        log(f"The two jobs' map frames differ by a global {gang:.1f} deg rotation"
            + (" (removed before classifying)." if applied else " (NOT removed)."))
    log(f"Flipped (> {THRESH_DEG:g} deg): {nf:,} = {p_all:.2%} overall; {p_ind:.2%} within the cleaned IND set "
        f"({nf_ind:,} particles). Non-flipped particles agree to {np.median(un):.2f} deg median "
        f"(95th pct {np.percentile(un, 95):.2f} deg).")
    if nf == 0:
        log("No ~180 deg disagreements: no evidence of a register swap between J486 and J487.")
    elif verdict_frame == "body":
        e = body_axis
        log(f"PSEUDO-2-FOLD CONFIRMED (in the cryoDRGN/cryoSPARC map frame): the flipped particles are rotated by "
            f"~{np.median(ang[flip_m]):.0f} deg about ONE map-frame axis u = {fmt_v(e)} (median spread "
            f"{stb['median_dev']:.1f} deg; {modes_b[0]['frac']:.0%} of flips within 15 deg), while their image-frame axes "
            f"scatter (lambda1 {stl['lambda1']:.2f}). J486 and J487 placed these particles with the two MINA53 protomers "
            f"swapped. The pose comparison alone cannot say which job is right for each particle.")
    elif verdict_frame == "lab":
        z_ang = np.degrees(np.arccos(min(1, abs(lab_axis[2]))))
        if z_ang < 15:
            log("The flips are ~180 deg IN-PLANE rotations (one image-frame axis along the beam), not one map-frame axis: "
                "a 2D in-plane ambiguity (psi vs psi+180). Under the cryoDRGN convention this is not a molecular "
                "pseudo-2-fold. Check the frame caveat in section 2.")
        else:
            log("The flips share one IMAGE-frame axis that is not the beam axis. This is unusual for a real particle "
                "symmetry. Check the frame caveat in section 2; under the swapped convention this would be the map-frame 2-fold.")
    elif verdict_frame == "both":
        log("The flip axes are tight in BOTH frames. That happens when most flipped particles are seen down the 2-fold "
            "(axis close to the beam), where a map-frame 2-fold and an in-plane 180 deg rotation are the same thing. "
            "See flip_vs_view.png.")
    else:
        log("NO single flip axis: the ~180 deg differences are about scattered axes, so this is NOT one pseudo-C2 swap. "
            "If several axis clusters are listed in section 4, there may be several pseudo-symmetries. Otherwise the "
            "orientations are simply poorly determined for these particles.")
    if nf:
        if assoc:
            log("Links between the flips and the cryoDRGN heterogeneity or metadata:")
            for a in assoc:
                log(f"  * {a}")
            log("  -> Part of the latent heterogeneity may be protomer-register mixing rather than biology. For example,")
            log("     MPP6 on 'protomer A' vs 'protomer B' looks like an occupancy change when the register is swapped.")
        else:
            log("The flips are NOT enriched in any UMAP region, landscape state or volume PC (by the thresholds used),")
            log("  so the cryoDRGN heterogeneity is unlikely to be driven by the register swap.")
        if verdict_frame in ("body", "both"):
            e = body_axis
            c = (D487 or 0) / 2 * (psz or 1)
            log("Next checks:")
            log(f"  1. Is the map really (pseudo)symmetric about u? In ChimeraX, with the consensus map as #1:")
            log(f"       volume copy #1 ; turn {e[0]:.4f},{e[1]:.4f},{e[2]:.4f} 180 models #2 center {c:.1f},{c:.1f},{c:.1f} coordinateSystem #1")
            log(f"       measure correlation #2 inMap #1 ; fitmap #2 inMap #1")
            log("     (the centre is the box centre in A. The axis is in map x,y,z; if the copy does not overlay, the map's axis")
            log("      order or hand may differ from cryoSPARC's pose frame, so treat this as a check, not a proof).")
            log("  2. Which register is right for the flipped particles? Back-project flipped_global_ind_in_ind.pkl twice with")
            log("     cryoDRGN backproject_voxel (or the backproject notebook): once with pose.pkl (J487) and once with")
            log("     pose_J486_in_J487_order.pkl (J486). Then compare each to the unflipped consensus, especially the MPP6 density.")
            log("  3. If both registers give MPP6-like density on different protomers, the dimer is close to C2. Consider")
            log("     C2 symmetry expansion followed by focused classification or local refinement on MPP6, and use")
            log("     register-consistent poses before re-training cryoDRGN.")
    S["verdict"] = dict(frame=verdict_frame, associations=assoc)
    S["elapsed_s"] = time.time() - t0

    def _js(o):
        if isinstance(o, dict):
            return {k: _js(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)):
            return [_js(v) for v in o]
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (np.floating, float)):
            return None if not np.isfinite(o) else float(o)
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, np.bool_):
            return bool(o)
        return o

    with open(os.path.join(OUT_DIR, "pc2_summary.json"), "w") as f:
        json.dump(_js(S), f, indent=1)
    with open(os.path.join(OUT_DIR, "pc2_report.txt"), "w") as f:
        f.write("\n".join(_REPORT) + "\n")
    log(f"\nWrote pc2_summary.json and pc2_report.txt to {OUT_DIR}  ({time.time() - t0:.0f} s)")
    return S


PC2 = run()
